# Dia 20 — Criando web apps

**🃏 Cartas:** 20 · 41 · 62 &nbsp;·&nbsp; **📘 Original:** *Day 20. Web apps creation*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 20 — O painel do caso

Às 10h chega um e-mail da Delegada Rita com cópia para três escrivães: *"Recebi mais duas ligações
de vítimas. Alguém consegue me dizer se a chave PIX que elas citaram já está no caso?"*

A Iara lê em voz alta e olha para você.

— Esta é a quarta vez nesta semana. A cada pergunta, alguém tem que te chamar, você abre o terminal,
roda um script, copia o resultado e manda por e-mail. Não escala. A equipe precisa de um **painel**:
uma página onde eles digitam a cidade ou a chave PIX e veem a resposta sozinhos.

Você pensa em frameworks, servidores, hospedagem. Ela já está balançando a cabeça.

— O Python traz um servidor web de fábrica. Para uso interno, na nossa rede, é suficiente. Mas
presta atenção numa coisa: **tudo o que alguém digita no formulário volta para a página**. Se você
devolver sem tratar, qualquer um escreve um `<script>` ali e o navegador do escrivão executa. Num
painel de investigação, isso é a pior vergonha possível.

Ela escreve no quadro, sublinhado duas vezes: **`html.escape` SEMPRE**.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-20-web-apps")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Rodar o web app original com **Streamlit** (localmente e, com algum esforço, no Colab).
- Entender o ciclo pedido → resposta HTTP (método, caminho, status, cabeçalhos, corpo).
- Escrever um servidor e um formulário de busca só com `http.server`.
- Ler parâmetros da URL com `urllib.parse.parse_qs`.
- Entender **XSS** e evitá-lo com `html.escape` (e uma Content-Security-Policy).
- Testar um web app automaticamente, sem navegador.

## 🧠 Conceito em 2 minutos

Um **web app** é um programa que fica esperando pedidos. Quando o navegador abre
`http://127.0.0.1:8001/?campo=cidade&q=contagem`, ele manda ao servidor:

```
GET /?campo=cidade&q=contagem HTTP/1.1
```

O servidor separa o **caminho** (`/`) da **consulta** (`campo=cidade&q=contagem`), faz a busca e
responde com um **status** (`200 OK`, `404 Not Found`), **cabeçalhos** (`Content-Type: text/html`) e
o **corpo** (o HTML).

| Peça | Streamlit (original) | Biblioteca padrão (BR) |
|---|---|---|
| Servidor | embutido, `streamlit run app.py` | `http.server.ThreadingHTTPServer` |
| Formulário | `st.text_input`, `st.button` | `<form method="get">` em HTML |
| Ler o que foi digitado | variável Python | `urllib.parse.parse_qs` |
| Escapar HTML | automático | **você**: `html.escape` |

**XSS (Cross-Site Scripting):** se alguém busca `<script>…</script>` e o servidor devolve o termo do
jeito que veio, o navegador **executa** aquele código, com acesso ao que o usuário vê. `html.escape`
troca `<` por `&lt;` e `>` por `&gt;`: o navegador mostra o texto em vez de executar.

**127.0.0.1 × 0.0.0.0:** o primeiro só aceita conexões do próprio computador; o segundo, de qualquer
máquina da rede. Comece sempre pelo 127.0.0.1.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/webapp.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-20-web-apps/original/webapp.py) | Página com título, caixa de texto e botão que mostra o texto digitado (Streamlit) |

**No seu computador** (o jeito fácil):

```bash
pip install streamlit
cd dia-20-web-apps/original
streamlit run webapp.py       # NÃO é "python webapp.py": o Streamlit tem o próprio comando
```

O terminal mostra uma *Local URL* (`http://localhost:8501`) e uma *Network URL*. Abra a primeira no
navegador. Para parar: `Ctrl+C`.

**No Colab** (sendo honesto: dá trabalho). O Streamlit precisa de um servidor rodando o tempo todo,
e o Colab não abre portas para a internet. O caminho que costuma funcionar:

```python
!pip install -q streamlit
!streamlit run original/webapp.py --server.port 8501 --server.headless true &>/content/st.log &
from google.colab import output
output.serve_kernel_port_as_window(8501)   # abre um link para a porta, só para você
```

Se o link abrir em branco, espere alguns segundos e recarregue. Túneis públicos (ngrok,
localtunnel, cloudflared) também funcionam, mas expõem o app para a internet inteira: não use com
dados de caso. Para estudar Streamlit de verdade, rode localmente ou use o Codespaces (Dia 0).

> 💡 No `webapp.py`, a variável `nickname = textInput.title()` é calculada e nunca usada. O autor
> conta, num artigo, como usou esse mesmo esqueleto para transformar o **Maigret** (busca de
> usernames) num web app.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q streamlit

### `original/webapp.py` — Um web app mínimo com Streamlit

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Um web app mínimo com Streamlit
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_20/webapp.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install streamlit · Rode com: streamlit run webapp.py (não com python)
# ──────────────────────────────────────────────────────────────────────────
# 💬 Streamlit transforma um script Python em página web. Cada clique RODA O SCRIPT DE NOVO.
import streamlit as st

# 💬 Título grande no topo da página.
st.title("Simple web app")


# 💬 Caixa de texto; o segundo argumento é o valor inicial.
textInput = st.text_input("Enter some text", "…")

# 💬 st.button devolve True só na rodada em que foi clicado.
if(st.button("Start")):
    # 💬 Esta variável é calculada mas nunca usada (resto de uma versão anterior, talvez).
    nickname = textInput.title()
    # 💬 Mostra o texto na página. O Streamlit escapa o HTML para você.
    st.text("You entered: "+textInput)

## 🇧🇷 Nível 2 — Camada Brasil

Só **biblioteca padrão**, 100% offline (dados do caso).

| Arquivo | O que faz |
|---|---|
| [`br/01_servidor_basico.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-20-web-apps/br/01_servidor_basico.py) | `BaseHTTPRequestHandler` com `do_GET`, status, cabeçalhos, 404 e log de cada pedido (porta 8000) |
| [`br/02_painel_do_caso.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-20-web-apps/br/02_painel_do_caso.py) | **O painel:** formulário de busca por cidade, chave PIX ou produto em `denuncias.csv`, resultados em tabela, `parse_qs`, `html.escape` e CSP (porta 8001) |

```bash
cd dia-20-web-apps
python br/02_painel_do_caso.py          # abra http://127.0.0.1:8001 · Ctrl+C para parar
OSINT_OFFLINE=1 python br/02_painel_do_caso.py   # modo teste
```

**Modo teste.** Com `OSINT_OFFLINE=1` (ou quando a saída não é um terminal, como no teste
automático), o script sobe o servidor numa *thread*, faz pedidos a si mesmo com `urllib`, imprime o
resultado e **encerra sozinho**. O painel testa inclusive um ataque: busca
`<script>alert('xss')</script>` e confere que voltou escapado.

**No Colab**, os scripts detectam o notebook, rodam o servidor em segundo plano (a célula não trava)
e chamam `google.colab.output.serve_kernel_port_as_window(porta)`, dentro de `try/except
ImportError` para não quebrar fora do Colab. Rodou a célula duas vezes e deu "porta ocupada"?
Troque `PORTA` ou reinicie o ambiente.

**O que você deve ver no caso:** busque `contagem` (4 denúncias), `f38b` no campo chave PIX (16
denúncias, a conta da Jéssica Prado Lima) e `poços` (3 denúncias: a busca sem acento acha também
"Pocos de Caldas").

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_servidor_basico.py` — Seu primeiro servidor web, só com a biblioteca padrão (http.server).

In [ ]:
%cd -q {PASTA_NB}
"""
Seu primeiro servidor web, só com a biblioteca padrão (http.server).

Carta 41 (rank A) — Jutsu do Painel

O que você aprende:
  • o que acontece quando o navegador pede uma página: requisição GET → resposta com status e HTML;
  • a escrever um "handler" com BaseHTTPRequestHandler e o método do_GET;
  • a escolher o endereço: 127.0.0.1 (só o seu computador) × 0.0.0.0 (toda a rede);
  • a testar o próprio servidor sem navegador, com urllib.

Como rodar (dentro da pasta dia-20-web-apps):
    python br/01_servidor_basico.py           # abra http://127.0.0.1:8000 · Ctrl+C para parar
    OSINT_OFFLINE=1 python br/01_servidor_basico.py   # modo teste: sobe, faz 1 pedido e encerra
"""

import os
import sys
import threading
import urllib.request
from datetime import datetime, timedelta, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

# ── Parâmetros ────────────────────────────────────────────────────────────
PORTA = 8000  #@param {type:"integer"}
ENDERECO = "127.0.0.1"  #@param ["127.0.0.1", "0.0.0.0"]

# Modo teste: o teste automático não pode ficar esperando você apertar Ctrl+C.
# Entra em modo teste com OSINT_OFFLINE=1 ou quando o script roda "sem terminal"
# (saída redirecionada, como no teste automático). No Colab, nunca.
EM_NOTEBOOK = "__file__" not in globals()
MODO_TESTE = os.environ.get("OSINT_OFFLINE") == "1" or (not EM_NOTEBOOK and not sys.stdout.isatty())

try:
    from zoneinfo import ZoneInfo
    FUSO = ZoneInfo("America/Sao_Paulo")
except Exception:
    FUSO = timezone(timedelta(hours=-3))


class Pagina(BaseHTTPRequestHandler):
    """Cada vez que chega um pedido, o servidor cria um objeto desta classe e chama do_GET."""

    def do_GET(self):
        # self.path é o caminho pedido: "/", "/favicon.ico", "/qualquer/coisa"...
        if self.path != "/":
            self.send_error(404, "Página não encontrada")   # resposta pronta de erro
            return
        agora = datetime.now(FUSO)
        corpo = f"""<!doctype html>
<html lang="pt-BR"><head><meta charset="utf-8"><title>NFA</title></head>
<body style="font-family: system-ui; max-width: 600px; margin: 3rem auto">
  <h1>🕵️ Núcleo de Fontes Abertas</h1>
  <p>Servidor no ar. Hora do servidor: <b>{agora:%d/%m/%Y %H:%M:%S}</b> (Brasília).</p>
  <p>Amanhã isto vira o painel do caso.</p>
</body></html>"""
        dados = corpo.encode("utf-8")              # a rede transporta BYTES, não texto
        self.send_response(200)                    # 200 = OK
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        self.end_headers()                         # linha em branco: acabaram os cabeçalhos
        self.wfile.write(dados)                    # o corpo da resposta

    def log_message(self, formato, *args):
        """Cada pedido vira uma linha no terminal: quem, quando, o quê."""
        print(f"📥 {self.client_address[0]} → {formato % args}")


# ── Subir o servidor ──────────────────────────────────────────────────────
# Porta 0 = "sistema, me dá uma porta livre qualquer" (útil no teste, para não dar conflito).
try:
    servidor = ThreadingHTTPServer((ENDERECO, 0 if MODO_TESTE else PORTA), Pagina)
except OSError as erro:
    # Acontece ao rodar a célula duas vezes no notebook: o servidor anterior ainda ocupa a porta.
    raise SystemExit(f"❌ Porta {PORTA} ocupada ({erro}). Troque PORTA ou reinicie o kernel.")
porta_real = servidor.server_address[1]

if MODO_TESTE:
    # Thread = o servidor roda "em paralelo", enquanto o resto do script continua.
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    with urllib.request.urlopen(f"http://127.0.0.1:{porta_real}/", timeout=10) as resposta:
        html_recebido = resposta.read().decode("utf-8")
        print(f"🧪 Modo teste: GET / → HTTP {resposta.status}, {len(html_recebido)} caracteres")
        print("   " + next(l.strip() for l in html_recebido.splitlines() if "<h1>" in l))
    servidor.shutdown()
    servidor.server_close()
    print("✅ Servidor respondeu e foi encerrado.")
else:
    print(f"🌐 Servidor no ar em http://{ENDERECO}:{porta_real}  {'(reinicie o ambiente para parar)' if EM_NOTEBOOK else '(Ctrl+C para parar)'}")
    if ENDERECO == "0.0.0.0":
        print("⚠️ 0.0.0.0 = qualquer computador da sua rede consegue abrir. Use só se precisar.")
    if EM_NOTEBOOK:
        # Num notebook, serve_forever() travaria a célula para sempre. Então o servidor roda
        # numa thread "de fundo" e a célula termina; o servidor continua vivo até o kernel reiniciar.
        threading.Thread(target=servidor.serve_forever, daemon=True).start()
        try:
            from google.colab import output       # só existe dentro do Google Colab
            output.serve_kernel_port_as_window(porta_real)
        except ImportError:
            print("   (Jupyter local: abra o endereço acima no navegador.)")
    else:
        try:
            servidor.serve_forever()
        except KeyboardInterrupt:
            print("\n👋 Servidor encerrado.")
            servidor.server_close()

### `br/02_painel_do_caso.py` — O painel do caso: um web app para a equipe buscar nas denúncias.

In [ ]:
%cd -q {PASTA_NB}
"""
O painel do caso: um web app para a equipe buscar nas denúncias.

Carta 41 (rank A) — Jutsu do Painel (parte 2)

O que você aprende:
  • a ler os parâmetros da URL (?campo=cidade&q=contagem) com urllib.parse (urlsplit + parse_qs);
  • a montar um formulário HTML que envia uma busca por GET;
  • a escapar TUDO o que vai para a página com html.escape, e por que isso evita XSS;
  • a buscar sem acento e sem maiúscula (a chave de comparação do Dia 16);
  • a testar o próprio app automaticamente, inclusive com um ataque de XSS.

Como rodar (dentro da pasta dia-20-web-apps):
    python br/02_painel_do_caso.py            # abra http://127.0.0.1:8001 · Ctrl+C para parar
    OSINT_OFFLINE=1 python br/02_painel_do_caso.py   # modo teste: sobe, faz pedidos e encerra
"""

import csv
import html
import os
import sys
import threading
import unicodedata
import urllib.parse
import urllib.request
from decimal import Decimal
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

# ── Parâmetros ────────────────────────────────────────────────────────────
PORTA = 8001  #@param {type:"integer"}
ENDERECO = "127.0.0.1"  #@param ["127.0.0.1", "0.0.0.0"]
MAX_RESULTADOS = 50  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
# Modo teste com OSINT_OFFLINE=1 ou sem terminal (teste automático). Ver br/01.
EM_NOTEBOOK = "__file__" not in globals()
MODO_TESTE = os.environ.get("OSINT_OFFLINE") == "1" or (not EM_NOTEBOOK and not sys.stdout.isatty())

# Campos que podem ser buscados: nome interno → rótulo no formulário.
# O nome e o CPF das vítimas NÃO estão aqui de propósito (minimização, LGPD).
CAMPOS = {"cidade": "Cidade", "chave_pix_destino": "Chave PIX", "produto": "Produto"}


def chave(texto):
    """Sem acento, minúsculo, espaços arrumados: 'POÇOS  de caldas' → 'pocos de caldas'."""
    sem = "".join(c for c in unicodedata.normalize("NFKD", texto) if not unicodedata.combining(c))
    return " ".join(sem.casefold().split())


def reais(valor):
    return "R$ " + f"{valor:,.2f}".replace(",", "§").replace(".", ",").replace("§", ".")


# ── 1. Carregar os dados UMA vez, quando o servidor sobe ──────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as arquivo:
    DENUNCIAS = list(csv.DictReader(arquivo, delimiter=";"))


def buscar(campo, termo):
    """Devolve as denúncias em que o termo aparece no campo (busca 'contém', sem acento)."""
    if campo not in CAMPOS or not termo.strip():
        return []
    alvo = chave(termo)
    return [d for d in DENUNCIAS if alvo in chave(d[campo])]


def pagina(campo, termo, resultados):
    """Monta o HTML. REGRA DE OURO: todo texto que veio de fora passa por html.escape."""
    e = html.escape
    opcoes = "".join(
        f'<option value="{e(nome)}"{" selected" if nome == campo else ""}>{e(rotulo)}</option>'
        for nome, rotulo in CAMPOS.items()
    )
    linhas = "".join(
        f"<tr><td>{e(d['id'])}</td><td>{e(d['data'])}</td><td>{e(d['cidade'])}</td>"
        f"<td>{e(d['produto'])}</td><td class='num'>{reais(Decimal(d['valor']))}</td>"
        f"<td><code>{e(d['chave_pix_destino'])}</code></td></tr>"
        for d in resultados[:MAX_RESULTADOS]
    )
    if termo:
        total = sum(Decimal(d["valor"]) for d in resultados)
        # O termo digitado aparece de volta na página: é AQUI que mora o risco de XSS.
        resumo = (f"<p><b>{len(resultados)}</b> denúncia(s) com <b>{e(CAMPOS.get(campo, '?'))}</b> "
                  f"contendo “<mark>{e(termo)}</mark>” · prejuízo {reais(total)}</p>")
        tabela = (f"<table><tr><th>ID</th><th>Data</th><th>Cidade</th><th>Produto</th><th>Valor</th>"
                  f"<th>Chave PIX</th></tr>{linhas}</table>" if resultados else "<p>Nada encontrado.</p>")
    else:
        resumo, tabela = f"<p>{len(DENUNCIAS)} denúncias carregadas. Faça uma busca.</p>", ""
    return f"""<!doctype html>
<html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Painel do caso — NFA</title>
<style>
 body {{ font-family: system-ui, sans-serif; max-width: 1000px; margin: 1.5rem auto; padding: 0 1rem; }}
 form {{ display: flex; gap: .5rem; flex-wrap: wrap; margin: 1rem 0; }}
 input, select, button {{ font-size: 1rem; padding: .4rem .6rem; }}
 table {{ border-collapse: collapse; width: 100%; font-size: .9rem; }}
 th, td {{ border-bottom: 1px solid #ddd; padding: .3rem .5rem; text-align: left; }}
 th {{ background: #1f5fa8; color: #fff; }} .num {{ text-align: right; white-space: nowrap; }}
 code {{ font-size: .8rem; }}
</style></head>
<body>
<h1>🕵️ Painel — Operação Vitrine Fantasma</h1>
<form method="get" action="/">
  <select name="campo">{opcoes}</select>
  <input name="q" value="{e(termo)}" placeholder="ex.: contagem, f38b, iphone" autofocus>
  <button>Buscar</button>
</form>
{resumo}
{tabela}
<p style="color:#777;font-size:.8rem">Dados fictícios de estudo · uso interno do NFA</p>
</body></html>"""


class Painel(BaseHTTPRequestHandler):
    def do_GET(self):
        # urlsplit separa "/?campo=cidade&q=belo" em caminho "/" e consulta "campo=cidade&q=belo".
        partes = urllib.parse.urlsplit(self.path)
        if partes.path != "/":
            self.send_error(404, "Página não encontrada")
            return
        # parse_qs devolve LISTAS (um parâmetro pode aparecer várias vezes): {"q": ["belo"]}.
        # Também já desfaz a codificação da URL: "belo+horizonte" → "belo horizonte".
        parametros = urllib.parse.parse_qs(partes.query)
        campo = parametros.get("campo", ["cidade"])[0]
        termo = parametros.get("q", [""])[0][:100]          # limita o tamanho da busca
        corpo = pagina(campo, termo, buscar(campo, termo)).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(corpo)))
        # Camada extra contra XSS: o navegador só executa scripts vindos deste próprio servidor.
        self.send_header("Content-Security-Policy", "default-src 'self'; style-src 'unsafe-inline'")
        self.end_headers()
        self.wfile.write(corpo)

    def log_message(self, formato, *args):
        print(f"📥 {self.client_address[0]} {formato % args}")


try:
    servidor = ThreadingHTTPServer((ENDERECO, 0 if MODO_TESTE else PORTA), Painel)
except OSError as erro:
    # Acontece ao rodar a célula duas vezes no notebook: o servidor anterior ainda ocupa a porta.
    raise SystemExit(f"❌ Porta {PORTA} ocupada ({erro}). Troque PORTA ou reinicie o kernel.")
porta_real = servidor.server_address[1]

if MODO_TESTE:
    # ── Modo teste: sobe numa thread, faz pedidos a si mesmo e encerra ────
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    base = f"http://127.0.0.1:{porta_real}/?"

    def pedir(campo, termo):
        url = base + urllib.parse.urlencode({"campo": campo, "q": termo})
        with urllib.request.urlopen(url, timeout=10) as resposta:
            return resposta.read().decode("utf-8")

    try:
        resposta = pedir("cidade", "contagem")
        achados = resposta.count("<tr><td>DEN-")
        print(f"🧪 Busca cidade='contagem' → {achados} denúncias na tabela")

        resposta = pedir("chave_pix_destino", "f38b")
        print(f"🧪 Busca chave='f38b'      → {resposta.count('<tr><td>DEN-')} denúncias na tabela")

        ataque = "<script>alert('xss')</script>"
        resposta = pedir("produto", ataque)
        seguro = ataque not in resposta and html.escape(ataque) in resposta
        print(f"🧪 Busca com <script>       → {'✅ escapado, não executa' if seguro else '❌ VULNERÁVEL'}")
        if not seguro:
            raise SystemExit("O painel devolveu o <script> sem escapar!")
    finally:
        servidor.shutdown()
        servidor.server_close()
    print("✅ Painel testado e encerrado.")
else:
    print(f"🌐 Painel em http://{ENDERECO}:{porta_real}  {'(reinicie o ambiente para parar)' if EM_NOTEBOOK else '(Ctrl+C para parar)'}")
    if EM_NOTEBOOK:
        # Num notebook, serve_forever() travaria a célula para sempre. Então o servidor roda
        # numa thread "de fundo" e a célula termina; o servidor continua vivo até o kernel reiniciar.
        threading.Thread(target=servidor.serve_forever, daemon=True).start()
        try:
            from google.colab import output       # só existe dentro do Google Colab
            output.serve_kernel_port_as_window(porta_real)
        except ImportError:
            print("   (Jupyter local: abra o endereço acima no navegador.)")
    else:
        try:
            servidor.serve_forever()
        except KeyboardInterrupt:
            print("\n👋 Painel encerrado.")
            servidor.server_close()

## ⚖️ Ética, LGPD e OPSEC

- **O painel é interno.** Use `127.0.0.1` ou, no máximo, a rede do NFA. Nada de túnel público para
  um painel com dados de vítimas. `http.server` não tem login nem HTTPS: não serve para produção.
- **Minimização na tela.** O painel não mostra nome nem CPF das vítimas: para saber se uma chave PIX
  já está no caso, ninguém precisa deles.
- **XSS e injeção.** Todo texto vindo de fora (busca, dados raspados, relatos de vítimas) passa por
  `html.escape` antes de ir para a página. E se um dia o painel consultar um banco, use parâmetros
  `?` (Dia 8), nunca f-string em SQL.
- **Log é dado pessoal também.** O servidor registra o IP e o que cada pessoa buscou. Isso é útil
  para auditoria, mas tem que ser guardado e apagado com o mesmo cuidado que o resto do caso.
- Um web app OSINT **aberto ao público** (como o Maigret + Streamlit do autor) vira ferramenta de
  qualquer pessoa, inclusive de quem quer perseguir alguém. Pense em quem vai usar antes de publicar.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Por que o painel passa o termo buscado por html.escape antes de mostrá-lo na página?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Para a busca ficar mais rápida", "B) Para que um <script> digitado apareça como texto e não seja executado (XSS)", "C) Para tirar os acentos", "D) Porque o http.server exige"]
correta = "B) Para que um <script> digitado apareça como texto e não seja executado (XSS)"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "html.escape troca < > & \" por entidades; o navegador mostra o texto em vez de interpretá-lo como HTML.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. O que urllib.parse.parse_qs('campo=cidade&q=belo+horizonte') devolve?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) {'campo': 'cidade', 'q': 'belo+horizonte'}", "B) {'campo': ['cidade'], 'q': ['belo horizonte']}", "C) ['cidade', 'belo horizonte']", "D) Um erro, por causa do '+'"]
correta = "B) {'campo': ['cidade'], 'q': ['belo horizonte']}"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Cada valor vem numa lista (um parâmetro pode se repetir) e o '+' é decodificado como espaço.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. Qual a diferença entre subir o servidor em 127.0.0.1 e em 0.0.0.0?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Nenhuma", "B) 127.0.0.1 só aceita conexões do próprio computador; 0.0.0.0 aceita de qualquer máquina da rede", "C) 0.0.0.0 é mais seguro", "D) 127.0.0.1 só funciona no Windows"]
correta = "B) 127.0.0.1 só aceita conexões do próprio computador; 0.0.0.0 aceita de qualquer máquina da rede"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Comece sempre pelo 127.0.0.1; abra para a rede só quando precisar e souber quem acessa.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 20: A API do caso

**Carta 62 · Rank S**

## A missão

O painel fez sucesso. Agora o setor de análise quer **puxar os dados direto para as planilhas e
scripts deles**, sem copiar da tela. Eles pedem uma **API**: endereços que devolvem JSON.

Escreva um servidor, só com `http.server`, que tenha:

1. `GET /api/denuncias` com filtros opcionais que se somam:
   `cidade` (contém, sem acento), `chave` (começa com), `de` e `ate` (datas `AAAA-MM-DD`).
   Resposta: `{"quantidade": N, "filtros": {...}, "denuncias": [...]}`.
2. `GET /api/resumo`: total de denúncias, total em reais, total por chave PIX e quantidade por mês.
3. `GET /`: uma página HTML curta explicando as rotas.
4. Qualquer outra rota: **404 em JSON** (`{"erro": ...}`), não uma página HTML.
5. As denúncias publicadas **não** podem conter `nome_vitima` nem `cpf`.
6. Modo teste (com `OSINT_OFFLINE=1`): sobe numa thread, testa todas as rotas a si mesmo e encerra.

## Critérios de rank S

- [ ] Só biblioteca padrão
- [ ] `Content-Type: application/json; charset=utf-8` nas rotas da API
- [ ] Valores em reais sem erro de arredondamento (`Decimal`, enviado como texto)
- [ ] Um `assert` garante que nome e CPF não aparecem na resposta
- [ ] O teste não trava: o script termina sozinho

## Dica

Datas no formato ISO (`2025-11-28`) podem ser comparadas **como texto**: `"2025-11-01" <= d <=
"2025-11-30"` funciona, porque a ordem alfabética é a ordem cronológica. Isso não vale para
`28/11/2025`!

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 20 — A API do caso.

Um servidor com três rotas, só com stdlib:
  /                      página simples explicando a API
  /api/denuncias?...     JSON com as denúncias filtradas (cidade, chave, de, ate) — SEM nome e CPF
  /api/resumo            JSON com totais por chave PIX e por mês
Com OSINT_OFFLINE=1 (ou sem terminal), sobe numa thread, testa as rotas a si mesmo e encerra.
"""

import csv
import json
import os
import sys
import threading
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
from collections import defaultdict
from decimal import Decimal
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

PORTA = 8002  #@param {type:"integer"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
EM_NOTEBOOK = "__file__" not in globals()
MODO_TESTE = os.environ.get("OSINT_OFFLINE") == "1" or (not EM_NOTEBOOK and not sys.stdout.isatty())

# Campos publicados pela API. Nome e CPF da vítima ficam de fora (minimização).
PUBLICOS = ["id", "data", "cidade", "uf", "valor", "produto", "chave_pix_destino"]


def chave(t):
    s = "".join(c for c in unicodedata.normalize("NFKD", t) if not unicodedata.combining(c))
    return " ".join(s.casefold().split())


with open(PASTA_CASO / "denuncias.csv", encoding="utf-8") as f:
    DENUNCIAS = [{k: d[k] for k in PUBLICOS} for d in csv.DictReader(f, delimiter=";")]


def filtrar(p):
    """p = dicionário de parse_qs. Todos os filtros são opcionais e se somam (E)."""
    cidade = chave(p.get("cidade", [""])[0])
    pix = p.get("chave", [""])[0].lower()
    de, ate = p.get("de", ["0000-00-00"])[0], p.get("ate", ["9999-99-99"])[0]
    return [d for d in DENUNCIAS
            if cidade in chave(d["cidade"]) and d["chave_pix_destino"].startswith(pix)
            and de <= d["data"] <= ate]                    # datas ISO comparam como texto!


def resumo():
    por_chave, por_mes = defaultdict(Decimal), defaultdict(int)
    for d in DENUNCIAS:
        por_chave[d["chave_pix_destino"]] += Decimal(d["valor"])
        por_mes[d["data"][:7]] += 1
    return {"total_denuncias": len(DENUNCIAS),
            "total_valor": str(sum(por_chave.values())),           # Decimal → texto (JSON não tem Decimal)
            "por_chave_pix": {k: str(v) for k, v in sorted(por_chave.items())},
            "por_mes": dict(sorted(por_mes.items()))}


class API(BaseHTTPRequestHandler):
    def responder(self, status, corpo, tipo):
        dados = corpo.encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", f"{tipo}; charset=utf-8")
        self.send_header("Content-Length", str(len(dados)))
        self.send_header("X-Content-Type-Options", "nosniff")   # o navegador não "adivinha" o tipo
        self.end_headers()
        self.wfile.write(dados)

    def json(self, status, objeto):
        self.responder(status, json.dumps(objeto, ensure_ascii=False, indent=2), "application/json")

    def do_GET(self):
        partes = urllib.parse.urlsplit(self.path)
        p = urllib.parse.parse_qs(partes.query)
        if partes.path == "/api/denuncias":
            achados = filtrar(p)
            self.json(200, {"quantidade": len(achados), "filtros": {k: v[0] for k, v in p.items()},
                            "denuncias": achados})
        elif partes.path == "/api/resumo":
            self.json(200, resumo())
        elif partes.path == "/":
            self.responder(200, "<!doctype html><meta charset='utf-8'><title>API NFA</title>"
                                "<h1>API do caso</h1><ul><li><a href='/api/resumo'>/api/resumo</a></li>"
                                "<li><a href='/api/denuncias?cidade=contagem'>/api/denuncias?cidade=contagem</a></li>"
                                "<li>/api/denuncias?chave=f38b&amp;de=2025-11-01&amp;ate=2025-11-30</li></ul>",
                           "text/html")
        else:
            self.json(404, {"erro": "rota inexistente", "rotas": ["/api/denuncias", "/api/resumo"]})

    def log_message(self, formato, *args):
        print(f"📥 {formato % args}")


try:
    servidor = ThreadingHTTPServer(("127.0.0.1", 0 if MODO_TESTE else PORTA), API)
except OSError as erro:
    raise SystemExit(f"❌ Porta {PORTA} ocupada ({erro}).")
porta = servidor.server_address[1]

if MODO_TESTE:
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    base = f"http://127.0.0.1:{porta}"

    def get_json(caminho):
        try:
            with urllib.request.urlopen(base + caminho, timeout=10) as r:
                return r.status, json.load(r)
        except urllib.error.HTTPError as erro:       # 404 também tem corpo JSON
            return erro.code, json.load(erro)

    try:
        st, r = get_json("/api/resumo")
        print(f"🧪 /api/resumo → {st}: {r['total_denuncias']} denúncias, R$ {r['total_valor']}")
        assert r["total_valor"] == "54877.40"

        st, r = get_json("/api/denuncias?" + urllib.parse.urlencode({"cidade": "juiz de fora"}))
        print(f"🧪 cidade='juiz de fora' → {r['quantidade']} denúncias")

        st, r = get_json("/api/denuncias?chave=f38b&de=2025-11-01&ate=2025-11-30")
        print(f"🧪 chave f38b em novembro → {r['quantidade']} denúncias")
        vazou = any("cpf" in d or "nome_vitima" in d for d in r["denuncias"])
        print(f"🧪 nome/CPF na resposta? {'❌ SIM' if vazou else '✅ não'}")
        assert not vazou

        st, r = get_json("/nao-existe")
        print(f"🧪 /nao-existe → {st} {r['erro']}")
        assert st == 404
    finally:
        servidor.shutdown()
        servidor.server_close()
    print("✅ API testada e encerrada.")
elif EM_NOTEBOOK:
    threading.Thread(target=servidor.serve_forever, daemon=True).start()
    print(f"🌐 API em http://127.0.0.1:{porta}")
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(porta)
    except ImportError:
        pass
else:
    print(f"🌐 API em http://127.0.0.1:{porta}  (Ctrl+C para parar)")
    try:
        servidor.serve_forever()
    except KeyboardInterrupt:
        servidor.server_close()

## 🔎 Inovação OSINT do dia

**Web app em Python que roda no navegador, sem servidor: stlite.** O **stlite** é um port do
Streamlit para **Pyodide** (Python compilado para WebAssembly). O app inteiro roda **dentro do
navegador de quem abre a página**, e os dados não saem do computador da pessoa. Para OSINT isso é
interessante: dá para distribuir uma ferramenta de análise como um único arquivo HTML, que funciona
até em computador sem Python instalado e não manda nada para servidor nenhum. Limitação: pacotes que
dependem de rede ou de código nativo nem sempre funcionam no Pyodide.

## 📚 Para ir além

- `http.server`: <https://docs.python.org/pt-br/3/library/http.server.html>
- `urllib.parse` (`urlsplit`, `parse_qs`, `urlencode`): <https://docs.python.org/pt-br/3/library/urllib.parse.html>
- OWASP: prevenção de XSS: <https://cheatsheetseries.owasp.org/cheatsheets/Cross_Site_Scripting_Prevention_Cheat_Sheet.html>
- Streamlit: <https://docs.streamlit.io/>
- stlite: <https://github.com/whitphx/stlite>
- MDN, Content-Security-Policy: <https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Headers/Content-Security-Policy>

## ✅ Checklist

- [ ] Rodei o `webapp.py` com `streamlit run`
- [ ] Subi o servidor básico e vi o log de cada pedido
- [ ] Usei o painel para buscar por cidade, chave PIX e produto
- [ ] Testei uma busca com `<b>oi</b>` e vi o texto aparecer escapado (não em negrito)
- [ ] Rodei os dois scripts em modo teste (`OSINT_OFFLINE=1`)
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 20](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)